# S04E05 — Food Warehouse (Magazyn Żywności)

Zadanie: Przygotowanie poprawnych zamówień żywnościowych dla wszystkich wskazanych miast.

Proces:
1. Pobierz zapotrzebowanie miast z `food4cities.json`
2. Zapytaj bazę SQLite o strukturę danych (creatorID, destination)
3. Wygeneruj podpisy SHA1 przez `signatureGenerator`
4. Utwórz osobne zamówienie dla każdego miasta
5. Uzupełnij zamówienia dokładnymi towarami
6. Wywołaj `done`

API: https://hub.ag3nts.org/verify (task: foodwarehouse)

Podejście agentowe: Claude orchestruje cały proces przez narzędzia API.

In [ ]:
import os, json, requests
from dotenv import load_dotenv
import anthropic

load_dotenv("../.env")
API_KEY = os.getenv("AI_DEVS_API_KEY")
ANTHROPIC_KEY = os.getenv("ANTHROPIC_API_KEY")

VERIFY_URL = "https://hub.ag3nts.org/verify"
TASK = "foodwarehouse"
FOOD_URL = "https://hub.ag3nts.org/dane/food4cities.json"

client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)

In [ ]:
# Funkcja pomocnicza — wywołanie API warehouse
def wh_api(answer: dict) -> dict:
    payload = {"apikey": API_KEY, "task": TASK, "answer": answer}
    r = requests.post(VERIFY_URL, json=payload)
    return r.json()

def wh_show(answer: dict) -> dict:
    result = wh_api(answer)
    print(f">> {json.dumps(answer)[:150]}")
    print(f"<< {json.dumps(result, ensure_ascii=False)[:400]}")
    return result

# Sprawdź dostępne narzędzia
wh_show({"tool": "help"})

In [ ]:
# Pobierz dane zapotrzebowania miast
print("Pobieram food4cities.json...")
r = requests.get(FOOD_URL)
food_data = r.json()
print("Zapotrzebowanie miast:")
print(json.dumps(food_data, indent=2, ensure_ascii=False))

## Definicja narzędzi agenta

In [ ]:
TOOLS = [
    {
        "name": "warehouse_tool",
        "description": (
            "Wywołuje narzędzie API magazynu żywności. "
            "Dostępne narzędzia (pole 'tool'): "
            "help, orders (z action: get/create/append/delete), "
            "database (z query: zapytanie SQL), "
            "signatureGenerator (z creatorID: liczba), "
            "reset, done."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "answer": {
                    "type": "object",
                    "description": "Obiekt answer z polem 'tool' i opcjonalnymi parametrami"
                }
            },
            "required": ["answer"]
        }
    }
]

def handle_tool(name: str, input_data: dict) -> str:
    if name == "warehouse_tool":
        answer = input_data["answer"]
        result = wh_api(answer)
        print(f"  >> {json.dumps(answer)[:200]}")
        print(f"  << {json.dumps(result, ensure_ascii=False)[:400]}")
        return json.dumps(result, ensure_ascii=False)
    return f"Nieznane narzędzie: {name}"

## Agent — pętla ReAct

In [ ]:
SYSTEM_PROMPT = """Jesteś agentem zarządzającym magazynem żywności. Musisz przygotować zamówienia dla wszystkich miast.

PROCES (wykonaj DOKŁADNIE w tej kolejności):

1. ROZPOZNANIE BAZY DANYCH:
   - Wywołaj: database z query="show tables" — poznaj strukturę bazy
   - Sprawdź zawartość każdej tabeli (SELECT * FROM ...)
   - Znajdź: creatorID (kto składa zamówienie) oraz kody destination (dla każdego miasta)

2. GENEROWANIE PODPISÓW:
   - Dla znalezionego creatorID wywołaj signatureGenerator
   - Zapamiętaj wygenerowany podpis SHA1

3. TWORZENIE ZAMÓWIEŃ:
   - Dla każdego miasta z food4cities.json utwórz OSOBNE zamówienie:
     - action: create
     - title: dowolny opis (np. "Dostawa dla [miasto]")
     - creatorID: znaleziony w bazie
     - destination: kod odpowiedni dla danego miasta (z bazy)
     - signature: wygenerowany podpis

4. UZUPEŁNIANIE ZAMÓWIEŃ:
   - Dla każdego zamówienia dodaj towary (action: append) dokładnie wg food4cities.json
   - Możesz użyć batch mode: items jako obiekt {"towar": ilosc, ...}
   - BEZ nadmiarów, BEZ braków!

5. WERYFIKACJA:
   - Sprawdź wszystkie zamówienia (action: get)
   - Upewnij się że każde miasto ma zamówienie z właściwymi towarami

6. FINALIZACJA:
   - Wywołaj: done

DANE MIAST:
"""

# Dodaj dane food4cities do promptu
food_context = json.dumps(food_data, indent=2, ensure_ascii=False)
full_system = SYSTEM_PROMPT + food_context + """

WAŻNE:
- Jeśli coś pójdzie nie tak, użyj reset i zacznij od nowa.
- Każde zamówienie musi mieć poprawny creatorID i signature.
- Jeśli w odpowiedzi pojawi się flaga (FLG:), wyświetl ją."""

messages = [
    {"role": "user", "content": "Rozpocznij tworzenie zamówień dla wszystkich miast. Zacznij od poznania struktury bazy danych."}
]

MAX_STEPS = 60
flag = None

for step in range(MAX_STEPS):
    print(f"\n{'='*60}")
    print(f"Krok {step + 1}")
    print('='*60)

    response = client.messages.create(
        model="claude-opus-4-6",
        max_tokens=8192,
        system=full_system,
        tools=TOOLS,
        messages=messages,
    )

    messages.append({"role": "assistant", "content": response.content})

    for block in response.content:
        if hasattr(block, 'text') and block.text:
            print(f"\n[Agent]: {block.text[:600]}")

    if response.stop_reason == "end_turn":
        print("\nAgent zakończył pracę.")
        break

    if response.stop_reason == "tool_use":
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"\n[Narzędzie]: {block.name}")
                result = handle_tool(block.name, block.input)

                if "FLG:" in result or "{FLG" in result:
                    flag = result

                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result
                })

        messages.append({"role": "user", "content": tool_results})

        if flag:
            print(f"\n{'*'*60}")
            print(f"FLAGA ZNALEZIONA: {flag}")
            print('*'*60)
            break

print("\nAgent zakończył pracę.")

In [ ]:
# Wyświetl flagę
if flag:
    print(f"Flaga: {flag}")
else:
    for msg in messages:
        content = str(msg.get('content', ''))
        if 'FLG' in content:
            print(f"Flaga w historii: {content[:500]}")
            break